Group XX | Members: [Student 1], [Student 2] | Question: Q1 | Stage: 4: cleaning and preprocessing

How this works: clean with explicit rules, log every operation, remove leakage from X, split first, then fit preprocessing on train only.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from pathlib import Path
import joblib
from src.config import RAW_FILE, STANDIN_FILE, INTERIM_DIR, PROCESSED_DIR, DOCS_DIR, TARGET_COL, LEAKAGE_COLS
from src.cleaning import to_string_df, clean_q1, add_time_features
from src.features import engineer_features
from src.logging_utils import CleaningLog
from src.pipeline import split_train_val_test, build_preprocessor, transform_split, add_low_yield_label

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

raw_path = RAW_FILE if RAW_FILE.exists() else STANDIN_FILE
raw = to_string_df(raw_path)
clog = CleaningLog()
clean = clean_q1(raw, clog)
clean = add_time_features(clean)
clean = engineer_features(clean)
clean.to_csv(INTERIM_DIR / 'maize_clean.csv', index=False)

train_df, val_df, test_df = split_train_val_test(clean)
q20 = add_low_yield_label(train_df, val_df, test_df)

pre, feature_cols = build_preprocessor(train_df)
joblib.dump({'preprocessor': pre, 'feature_cols': feature_cols, 'q20_threshold': q20}, PROCESSED_DIR / 'preprocessor.joblib')

Xtr = transform_split(pre, train_df, feature_cols)
Xva = transform_split(pre, val_df, feature_cols)
Xte = transform_split(pre, test_df, feature_cols)

for src, X, name in [(train_df,Xtr,'train'),(val_df,Xva,'validation'),(test_df,Xte,'test')]:
    full = pd.concat([src[['farm_id', TARGET_COL, 'low_yield_risk']].reset_index(drop=True), X.reset_index(drop=True)], axis=1)
    full.to_csv(PROCESSED_DIR / f'{name}_model_ready.csv', index=False)
    src.to_csv(PROCESSED_DIR / f'{name}_clean_unscaled.csv', index=False)

clog.save(DOCS_DIR / 'cleaning_log.csv', DOCS_DIR / 'cleaning_log.md')

# leak check + split disjoint
s1,s2,s3 = set(train_df['farm_id']), set(val_df['farm_id']), set(test_df['farm_id'])
print('Split shapes:', train_df.shape, val_df.shape, test_df.shape)
print('farm_id overlaps:', len(s1&s2), len(s1&s3), len(s2&s3))
print('Low-yield threshold (train 20th percentile):', q20)
print('Leakage columns removed from X:', LEAKAGE_COLS)


/home/runner/work/ML-Learning/ML-Learning/src/cleaning.py:78: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)


Split shapes: (834, 38) (179, 38) (179, 38)
farm_id overlaps: 0 0 0
Low-yield threshold (train 20th percentile): 1511.6019999999999
Leakage columns removed from X: ['harvest_weight_kg']
